In [ ]:
import os
from pathlib import Path

import pandas as pd
import numpy as np
from plotly import graph_objects as go
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown
import xarray as xr
import oq_wrapper as oqw

import nn_gmm as nng
import ml_tools as mlt
import seismic_hazard_analysis as sha

In [ ]:
magnitudes = np.arange(4.5, 7.5, 0.01)
# rrup_values = np.logspace(np.log10(0.1), np.log10(500), num=250, base=10.0)
rrup_values = np.linspace(0.1, 500, num=250)

mag_values, rrup_values = np.meshgrid(magnitudes, rrup_values)

In [ ]:
rupture_df = pd.DataFrame(data=[mag_values.ravel(), rrup_values.ravel()], index=["mag", "rrup"]).T
rupture_df["ztor"] = 60
rupture_df["vs30"] = 500
rupture_df["backarc"] = False
rupture_df["z1pt0"] = oqw.estimations.chiou_young_08_calc_z1p0(rupture_df["vs30"])
rupture_df["z2pt5"] = oqw.estimations.chiou_young_08_calc_z2p5(rupture_df["vs30"])

In [ ]:
results_df = oqw.run_gmm(oqw.constants.GMM.K_20, oqw.constants.TectType.SUBDUCTION_SLAB, rupture_df, "PGV")
pgv_values = np.exp(results_df["PGV_mean"].values)

In [ ]:
pgv_threshold = 1
tmp_resuts_df = results_df[["PGV_mean"]].copy()
tmp_resuts_df["mag"] = rupture_df["mag"]
tmp_resuts_df["rrup"] = rupture_df["rrup"]

In [ ]:
mag_rrup_values = tmp_resuts_df.groupby("mag")[["PGV_mean", "rrup"]].apply(lambda x: x.iloc[mlt.array_utils.find_nearest(x["PGV_mean"].values, np.log(pgv_threshold))]["rrup"])
mag_rrup_values.loc[mag_rrup_values < 100] = 100

In [ ]:
fig = plt.figure(figsize=(16, 8))

plt.pcolormesh(
    magnitudes,
    rrup_values,
    pgv_values.reshape(rrup_values.shape),
    cmap="Blues",   
    # vmin=0,
    # vmax=1.0,
    norm="log",
)

plt.plot(mag_rrup_values.index, mag_rrup_values.values, color="red", linewidth=3, label=f"PGV = {pgv_threshold}")

plt.colorbar(label="PGV (cm/s)")
plt.xlabel("Magnitude")
plt.ylabel("Rrup (m)")

In [ ]:
# Save
#mag_rrup_values.to_csv("/Users/claudy/dev/work/data/nn_gmm/results/ds_simulations/mag_rrup_pgv_1_minRrup_100.csv", index_label="mag", header=["rrup"])